# Biomedical Drift Experiment


In [ ]:
import os
import sys
from collections import deque
from dataclasses import dataclass
from pathlib import Path
import hashlib
import json
import math
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

PROJECT_DIR = Path.home() / "experiments" / "BioDrift"
ADAZOR_SRC = PROJECT_DIR / 'adazor' / 'src'
if str(ADAZOR_SRC) not in sys.path:
    sys.path.append(str(ADAZOR_SRC))

try:
    from adazor.detectors import ZTestDriftDetector
except SyntaxError:
    import scipy.stats as st

    class ZTestDriftDetector:
        # Fallback with the same reset/detect interface as adazor.
        def __init__(self, alpha=0.05):
            self.alpha = float(alpha)
            self.is_built = False
            self.tot_orig = 0
            self.n_orig = 0

        def reset(self, outliers):
            outs = np.fromiter((bool(o) for o in outliers), dtype=bool)
            self.tot_orig = int(outs.sum())
            self.n_orig = int(len(outs))
            if self.n_orig == 0:
                raise ValueError('reset requires a non-empty baseline window')
            self.is_built = True

        def detect(self, outliers):
            if not self.is_built:
                raise RuntimeError('ZTestDriftDetector has not been reset.')
            outs = np.fromiter((bool(o) for o in outliers), dtype=bool)
            n = int(len(outs))
            if n == 0:
                return False
            p_outliers = int(outs.sum()) / n
            p_orig = self.tot_orig / self.n_orig
            threshold = st.norm().ppf(1 - self.alpha / 2) / math.sqrt(2 * n)
            return bool(abs(p_outliers - p_orig) > threshold)

        def message(self):
            return 'Z-test Drift Detection (alpha={})'.format(self.alpha)

RESULTS_DIR = PROJECT_DIR / 'biomedical_experiment_results4'
RESULTS_DIR.mkdir(exist_ok=True)
PYSTOW_HOME = PROJECT_DIR / '.pystow'
PYSTOW_HOME.mkdir(exist_ok=True)
os.environ.setdefault('PYSTOW_HOME', str(PYSTOW_HOME))

TEMPORAL_TRIPLES_PATH = PROJECT_DIR / 'filtered_drifting_edgesCASE1.csv'
if not TEMPORAL_TRIPLES_PATH.exists():
    TEMPORAL_TRIPLES_PATH = PROJECT_DIR / 'filtered_drifting_edgesCASE1.tsv'

INITIAL_CUTOFF_YEAR = 2002
TARGET_TYPE = 'Gene - causes_or_contributes_to_condition - Disease'

EMBEDDING_DIM = 32
EMBEDDING_RANDOM_STATE = 42
EMBEDDING_EPOCHS = 2
EMBEDDING_BATCH_SIZE = 2048
WINDOW_SIZE = 250
NU = 0.1
GAMMA = 0.5
ALPHA = 0.025
STANDARDIZE = False
KEEP_FRACTION_AFTER_DRIFT = 0.10
EMBEDDING_SHIFT_RETRAIN_THRESHOLD = 0.10
PROGRESS_FILE = PROJECT_DIR / 'run_progress4.json'
PROGRESS_LOG_FILE = PROJECT_DIR / 'run_progress4.log'


def write_progress(stage, iteration, total=None, drifts=0, refreshes=0, unseen_all=0, composition='', detail=''):
    payload = {
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
        'stage': stage,
        'detail': detail,
        'composition': composition,
        'iteration': int(iteration),
        'total': int(total) if total is not None else None,
        'percent': round((100.0 * iteration / total), 2) if total else None,
        'drifts': int(drifts),
        'refreshes': int(refreshes),
        'unseen_all': int(unseen_all),
    }
    with PROGRESS_FILE.open('w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)
    with PROGRESS_LOG_FILE.open('a', encoding='utf-8') as f:
        f.write(json.dumps(payload) + '\n')

print('Project:', PROJECT_DIR)
print('Temporal triples:', TEMPORAL_TRIPLES_PATH)
print('PyStow home:', PYSTOW_HOME)
print('Target type:', TARGET_TYPE)
print('Progress file:', PROGRESS_FILE)
print('Progress log file:', PROGRESS_LOG_FILE)


In [ ]:
def read_table_auto(path, usecols=None, nrows=None, chunksize=None):
    # Read CSV/TSV inputs without loading generated embedding outputs.
    sep = '\t' if path.suffix.lower() == '.tsv' else ','
    return pd.read_csv(path, sep=sep, usecols=usecols, nrows=nrows, chunksize=chunksize)


def infer_event_year(df):
    # Prefer explicit year, otherwise derive a year from timestamp text.
    if 'year' in df.columns:
        year = pd.to_numeric(df['year'], errors='coerce')
    else:
        year = pd.Series(np.nan, index=df.index, dtype='float64')
    timestamp_year = pd.to_numeric(df['timestamp'].astype(str).str.extract(r'(\d{4})', expand=False), errors='coerce')
    return year.fillna(timestamp_year)


def sort_temporal_triples(df):
    # Sort static rows first, then future rows by event time and row order.
    numeric_ts = pd.to_numeric(df['timestamp'].where(df['timestamp'].ne(''), '0'), errors='coerce').fillna(0)
    sorted_df = df.assign(_missing_ts=df['timestamp'].eq(''), _numeric_ts=numeric_ts).sort_values(
        by=['_missing_ts', 'event_year', '_numeric_ts', 'row_number'],
        ascending=[False, True, True, True],
        kind='stable',
    )
    sorted_df = sorted_df.drop(columns=['_missing_ts', '_numeric_ts']).reset_index(drop=True)
    sorted_df['stream_position'] = np.arange(1, len(sorted_df) + 1)
    return sorted_df


def load_temporal_triples(path):
    # Load only temporal KG triples; embeddings are generated later from KG state.
    required_cols = ['subject', 'object', 'edge_type', 'timestamp'] #
    sample = read_table_auto(path, nrows=0)
    optional_cols = [c for c in ['year', 'month', 'day', 'type'] if c in sample.columns]
    df = read_table_auto(path, usecols=required_cols + optional_cols)
    df['row_number'] = np.arange(1, len(df) + 1)
    df['timestamp'] = df['timestamp'].fillna('').astype(str)
    if 'type' not in df.columns:
        df['type'] = df['subject'].map(lambda _: '')
    df['event_year'] = infer_event_year(df)
    return sort_temporal_triples(df)


def build_initial_knowledge_graph(triples_df, cutoff_year=INITIAL_CUTOFF_YEAR, target_type=None):
    # Initial KG uses only triples known through the cutoff year.
    initial_mask = triples_df['event_year'].isna() | (triples_df['event_year'] <= cutoff_year)
    initial_kg = triples_df.loc[initial_mask].copy().reset_index(drop=True)
    future_df = triples_df.loc[~initial_mask].copy().reset_index(drop=True)
    if target_type is None:
        initial_stream = initial_kg.copy().reset_index(drop=True)
        future_stream = future_df.copy().reset_index(drop=True)
    else:
        initial_stream = initial_kg[initial_kg['type'].eq(target_type)].copy().reset_index(drop=True)
        future_stream = future_df[future_df['type'].eq(target_type)].copy().reset_index(drop=True)
    lookup = build_lookup_sets(initial_kg)
    return initial_kg, initial_stream, future_stream, lookup


def build_lookup_sets(kg_df):
    # Store elements known in the current KG state.
    return {
        'nodes': set(kg_df['subject']).union(set(kg_df['object'])),
        'edge_types': set(kg_df['edge_type']),
    }


def triple_has_unseen_elements(row, lookup):
    # Check before adding the row to the KG.
    return (
        row['subject'] not in lookup['nodes']
        or row['object'] not in lookup['nodes']
        or row['edge_type'] not in lookup['edge_types']
    )


def update_knowledge_graph(kg_df, rows, lookup):
    # Add every incoming temporal triple to the current KG.
    rows_df = pd.DataFrame(rows) if not isinstance(rows, pd.DataFrame) else rows.copy()
    if rows_df.empty:
        return kg_df, lookup
    keep_cols = [c for c in kg_df.columns if c in rows_df.columns]
    rows_df = rows_df[keep_cols].copy()
    kg_df = pd.concat([kg_df, rows_df], ignore_index=True)
    lookup['nodes'].update(rows_df['subject'])
    lookup['nodes'].update(rows_df['object'])
    lookup['edge_types'].update(rows_df['edge_type'])
    return kg_df, lookup


In [3]:
@dataclass
class EmbeddingState:
    node_embeddings: dict
    edge_type_embeddings: dict
    node_dim: int
    edge_dim: int
    kg_rows: int


def normalize_vector(vec):
    vec = np.asarray(vec, dtype=np.float64)
    norm = np.linalg.norm(vec)
    if norm == 0:
        return vec
    return vec / norm


def _compute_predicate_embeddings_from_transe_translation(kg_df, node_embeddings, embedding_size):
    predicate_vectors = {}
    grouped = kg_df[['subject', 'edge_type', 'object']].dropna().groupby('edge_type', sort=True)
    for predicate, rows in grouped:
        diffs = []
        for row in rows.itertuples(index=False):
            subject_vec = node_embeddings.get(str(row.subject))
            object_vec = node_embeddings.get(str(row.object))
            if subject_vec is not None and object_vec is not None:
                diffs.append(object_vec - subject_vec)
        if diffs:
            predicate_vectors[str(predicate)] = normalize_vector(np.mean(diffs, axis=0))
        else:
            predicate_vectors[str(predicate)] = np.zeros(embedding_size, dtype=np.float64)
    return predicate_vectors


def _extract_pykeen_embeddings(model, labels):
    representation = model
    if not hasattr(representation, '_embeddings') and hasattr(representation, '__getitem__'):
        representation = representation[0]
    if not hasattr(representation, '_embeddings'):
        raise AttributeError('PyKEEN representation does not expose an embedding weight matrix.')
    weights = representation._embeddings.weight.detach().cpu().numpy()
    return {str(label): normalize_vector(weights[i]) for i, label in enumerate(labels)}


def generate_transe_embeddings_from_kg(kg_df, embedding_size=32, random_state=42):
    # Generate TransE embeddings only from the KG known at the current time.
    try:
        from pykeen.models import TransE
        from pykeen.training import SLCWATrainingLoop
        from pykeen.triples import TriplesFactory
    except Exception as exc:
        raise RuntimeError(
            'PyKEEN TransE embeddings are unavailable in this environment. '
            'Install compatible pykeen dependencies and ensure PYSTOW_HOME is writable.'
        ) from exc

    if kg_df.empty:
        raise ValueError('Cannot generate embeddings from an empty KG.')

    #nodes = pd.Index(pd.concat([kg_df['subject'], kg_df['object']], ignore_index=True).dropna().astype(str).unique())
    #nodes_df = pd.DataFrame({'name': sorted(nodes.tolist()), 'type': 'unknown'})
    edges_df = kg_df[['subject', 'edge_type', 'object']].dropna().copy()
    edges_df = edges_df.rename(columns={'edge_type': 'predicate'})
    edges_df[['subject', 'predicate', 'object']] = edges_df[['subject', 'predicate', 'object']].astype(str)
    triples = edges_df[['subject', 'predicate', 'object']].to_numpy(dtype=str)
    triples_factory = TriplesFactory.from_labeled_triples(triples)

    model = TransE(
        triples_factory=triples_factory,
        embedding_dim=embedding_size,
        random_seed=random_state,
    )
    training_loop = SLCWATrainingLoop(model=model, triples_factory=triples_factory)
    training_loop.train(
        triples_factory=triples_factory,
        num_epochs=EMBEDDING_EPOCHS,
        batch_size=min(EMBEDDING_BATCH_SIZE, max(32, len(triples))),
        use_tqdm=False,
        use_tqdm_batch=False,
        pin_memory=False,
        num_workers=0,
    )

    node_labels = list(triples_factory.entity_to_id.keys())
    relation_labels = list(triples_factory.relation_to_id.keys())
    node_embeddings = _extract_pykeen_embeddings(model.entity_representations, node_labels)
    try:
        edge_type_embeddings = _extract_pykeen_embeddings(model.relation_representations, relation_labels)
    except Exception:
        edge_type_embeddings = _compute_predicate_embeddings_from_transe_translation(
            edges_df.rename(columns={'predicate': 'edge_type'}),
            node_embeddings,
            embedding_size,
        )

    node_embeddings = {node: np.asarray(vec, dtype=np.float64) for node, vec in node_embeddings.items()}
    edge_type_embeddings = {edge_type: normalize_vector(vec) for edge_type, vec in edge_type_embeddings.items()}
    return EmbeddingState(
        node_embeddings=node_embeddings,
        edge_type_embeddings=edge_type_embeddings,
        node_dim=embedding_size,
        edge_dim=embedding_size,
        kg_rows=int(len(kg_df)),
    )


In [ ]:

def create_hadamard_features(row, embedding_state):
    # Compose with embeddings valid for the current KG time.
    s = embedding_state.node_embeddings[row['subject']]
    p = embedding_state.edge_type_embeddings[row['edge_type']]
    o = embedding_state.node_embeddings[row['object']]
    return s * p * o


def create_concatenation_features(row, embedding_state):
    # Concatenate subject, predicate, and object embeddings.
    s = embedding_state.node_embeddings[row['subject']]
    p = embedding_state.edge_type_embeddings[row['edge_type']]
    o = embedding_state.node_embeddings[row['object']]
    return np.concatenate([s, p, o])


def feature_columns(dim):
    return [f'f{i}' for i in range(dim)]


def row_to_feature_record(row, embedding_state, compose_fn, feature_cols):
    # Convert one raw triple row into one embedded feature record.
    features = compose_fn(row, embedding_state)
    record = {col: row[col] for col in row.index if col not in ['year', 'month', 'day']}
    for col, value in zip(feature_cols, features):
        record[col] = float(value)
    return record


def rows_to_feature_frame(rows_df, embedding_state, compose_fn, feature_cols):
    # Rebuild feature vectors from raw triples whenever embeddings refresh.
    records = [row_to_feature_record(row, embedding_state, compose_fn, feature_cols) for _, row in rows_df.iterrows()]
    return pd.DataFrame(records)


def build_feature_matrix(df, feature_cols, scaler=None, fit=False):
    X = df.loc[:, feature_cols].to_numpy(dtype=np.float64)
    if scaler is None:
        return X, None
    if fit:
        return scaler.fit_transform(X), scaler
    return scaler.transform(X), scaler


def outlier_flags_from_ocsvm(ocsvm, X):
    # OCSVM predicts -1 for outliers.
    return ocsvm.predict(X) == -1


def train_classifier(reference_features_df, feature_cols, nu=NU, gamma=GAMMA, standardize=STANDARDIZE):
    # Train OCSVM on the current reference feature window.
    scaler = StandardScaler() if standardize else None
    X_ref, scaler = build_feature_matrix(reference_features_df, feature_cols, scaler=scaler, fit=True)
    ocsvm = OneClassSVM(kernel='rbf', nu=nu, gamma=gamma)
    ocsvm.fit(X_ref)
    flags = outlier_flags_from_ocsvm(ocsvm, X_ref)
    labeled = reference_features_df.copy()
    labeled['ocsvm_prediction'] = np.where(flags, -1, 1)
    labeled['ocsvm_label'] = np.where(flags, 'outlier', 'normal')
    labeled['outlier_flag'] = flags.astype(int)
    return labeled, ocsvm, scaler, flags


def retrain_reference_model(current_reference_raw_df, current_embedding_state, compose_fn, feature_cols, detector, nu=NU, gamma=GAMMA, standardize=STANDARDIZE, iteration=None, total=None, composition_name=''):
    # Refresh the OCSVM reference using the current raw reference rows and embeddings.
    if iteration is not None:
        write_progress('retraining_ocsvm_start', iteration, total, composition=composition_name)
    reference_features = rows_to_feature_frame(current_reference_raw_df, current_embedding_state, compose_fn, feature_cols)
    _, new_ocsvm, new_scaler, new_reference_flags = train_classifier(
        reference_features,
        feature_cols,
        nu=nu,
        gamma=gamma,
        standardize=standardize,
    )
    detector.reset(new_reference_flags)
    if iteration is not None:
        write_progress('retraining_ocsvm_done', iteration, total, composition=composition_name)
    return reference_features, new_ocsvm, new_scaler, new_reference_flags


def build_labeled_window_snapshot(window_features_df, outlier_flags, drift_event_id, drift_stream_position):
    # Store the feature window that triggered drift and its labels.
    snapshot = window_features_df.copy().reset_index(drop=True)
    snapshot['ocsvm_prediction'] = np.where(outlier_flags, -1, 1)
    snapshot['ocsvm_label'] = np.where(outlier_flags, 'outlier', 'normal')
    snapshot['outlier_flag'] = np.asarray(outlier_flags, dtype=int)
    snapshot['drift_event_id'] = int(drift_event_id)
    snapshot['drift_stream_position'] = int(drift_stream_position)
    return snapshot


def _shared_embedding_delta_vectors(old_embeddings, new_embeddings):
    # Compare only elements that exist in both embedding states.
    shared_keys = sorted(set(old_embeddings).intersection(new_embeddings))
    if not shared_keys:
        return shared_keys, np.empty((0, 0), dtype=np.float64), np.empty((0,), dtype=np.float64)
    deltas = np.vstack([
        np.asarray(new_embeddings[key], dtype=np.float64) - np.asarray(old_embeddings[key], dtype=np.float64)
        for key in shared_keys
    ])
    distances = np.linalg.norm(deltas, axis=1)
    return shared_keys, deltas, distances

###for the center radius way 
def build_center_radius_shift_record(row, composition_name, iteration, trigger_reason, is_drift, threshold, kg_rows):
    # Keep legacy metric columns as NaN/False so existing plots and exports still run.
    return {
        'iteration': int(iteration),
        'stream_position': int(row['stream_position']),
        'row_number': int(row['row_number']),
        'timestamp': row['timestamp'],
        'type': row['type'],
        'subject': row['subject'],
        'edge_type': row['edge_type'],
        'object': row['object'],
        'composition': composition_name,
        'trigger_reason': trigger_reason,
        'is_drift': bool(is_drift),
        'avg_euclidean_distance': np.nan,
        'bounding_box_width': np.nan,
        'mad': np.nan,
        'max_element_distance': np.nan,
        'max_coordinate_spread': np.nan,
        'min_coordinate_spread': np.nan,
        'decision_distance_max_spread': np.nan,
        'decision_distance_min_spread': np.nan,
        'shared_nodes': 0,
        'shared_edge_types': 0,
        'compared_elements': 0,
        'distance_threshold': float(threshold),
        'threshold_exceeded_avg': False,
        'threshold_exceeded_max_spread': False,
        'threshold_exceeded_min_spread': False,
        'retrained_ocsvm': bool(is_drift),
        'embedding_kg_rows': int(kg_rows),
    }
#for the distance way between new and old embeddings 
def build_embedding_shift_record(old_state, new_state, row, composition_name, iteration, trigger_reason, is_drift, threshold):
    # Summarize how much shared element embeddings moved after regeneration.
    shared_nodes, node_deltas, node_distances = _shared_embedding_delta_vectors(
        old_state.node_embeddings,
        new_state.node_embeddings,
    )
    shared_edge_types, edge_deltas, edge_distances = _shared_embedding_delta_vectors(
        old_state.edge_type_embeddings,
        new_state.edge_type_embeddings,
    )

    delta_parts = [part for part in [node_deltas, edge_deltas] if part.size]
    distance_parts = [part for part in [node_distances, edge_distances] if part.size]
    if delta_parts:
        all_deltas = np.vstack(delta_parts)
        all_distances = np.concatenate(distance_parts)
        avg_distance = float(np.mean(all_distances))
        bounding_box_width = float(np.linalg.norm(all_deltas.max(axis=0) - all_deltas.min(axis=0)))
        mad = float(np.mean(np.abs(all_distances - avg_distance)))
        max_element_distance = float(np.max(all_distances))
        coordinate_spreads = all_deltas.max(axis=0) - all_deltas.min(axis=0)
        max_coordinate_spread = float(np.max(coordinate_spreads))
        min_coordinate_spread = float(np.min(coordinate_spreads))
        if max_coordinate_spread == 0.0:
            decision_distance_max_spread = np.inf if max_element_distance > 0 else 0.0
        else:
            decision_distance_max_spread = float(max_element_distance / max_coordinate_spread)
        if min_coordinate_spread == 0.0:
            decision_distance_min_spread = np.inf if max_element_distance > 0 else 0.0
        else:
            decision_distance_min_spread = float(max_element_distance / min_coordinate_spread)
    else:
        avg_distance = np.nan
        bounding_box_width = np.nan
        mad = np.nan
        max_element_distance = np.nan
        max_coordinate_spread = np.nan
        min_coordinate_spread = np.nan
        decision_distance_max_spread = np.nan
        decision_distance_min_spread = np.nan

    threshold_exceeded_avg = bool(np.isfinite(avg_distance) and avg_distance > threshold)
    threshold_exceeded_max_spread = bool(
        np.isfinite(decision_distance_max_spread) and decision_distance_max_spread > threshold
    )
    threshold_exceeded_min_spread = bool(
        np.isfinite(decision_distance_min_spread) and decision_distance_min_spread > threshold
    )
    #threshold_exceeded = threshold_exceeded_avg
    return {
        'iteration': int(iteration),
        'stream_position': int(row['stream_position']),
        'row_number': int(row['row_number']),
        'timestamp': row['timestamp'],
        'type': row['type'],
        'subject': row['subject'],
        'edge_type': row['edge_type'],
        'object': row['object'],
        'composition': composition_name,
        'trigger_reason': trigger_reason,
        'is_drift': bool(is_drift),
        'avg_euclidean_distance': avg_distance,
        'bounding_box_width': bounding_box_width,
        'mad': mad,
        'max_element_distance': max_element_distance,
        'max_coordinate_spread': max_coordinate_spread,
        'min_coordinate_spread': min_coordinate_spread,
        'decision_distance_max_spread': decision_distance_max_spread,
        'decision_distance_min_spread': decision_distance_min_spread,
        'shared_nodes': int(len(shared_nodes)),
        'shared_edge_types': int(len(shared_edge_types)),
        'compared_elements': int(len(shared_nodes) + len(shared_edge_types)),
        'distance_threshold': float(threshold),
        #'threshold_exceeded': bool(threshold_exceeded),
        'threshold_exceeded_avg': bool(threshold_exceeded_avg),
        'threshold_exceeded_max_spread': bool(threshold_exceeded_max_spread),
        'threshold_exceeded_min_spread': bool(threshold_exceeded_min_spread),
        'retrained_ocsvm': bool(is_drift),
        'embedding_kg_rows': int(new_state.kg_rows),
    }


def build_avg_distance_trace(shift_df):
    # Keep a compact trace for plotting mean embedding movement over time.
    if shift_df.empty:
        return pd.DataFrame(columns=['iteration', 'avg_euclidean_distance', 'is_drift', 'trigger_reason', 'composition'])
    cols = ['iteration', 'avg_euclidean_distance', 'is_drift', 'trigger_reason', 'composition']
    return shift_df.loc[:, cols].copy()


def save_metric_over_time_plot(trace_df, metric_col, output_path, title, y_label):
    # Plot a saved embedding-shift metric over algorithm iterations.
    fig, ax = plt.subplots(figsize=(10, 4.5))
    if trace_df.empty:
        ax.set_title(title)
        ax.set_xlabel('Iteration')
        ax.set_ylabel(y_label)
        ax.text(0.5, 0.5, 'No embedding refresh events', ha='center', va='center', transform=ax.transAxes)
    else:
        ordered = trace_df.sort_values('iteration').reset_index(drop=True)
        ax.step(ordered['iteration'], ordered[metric_col], where='post', color='steelblue', linewidth=1.5)
        non_drift = ordered[~ordered['is_drift']]
        drift = ordered[ordered['is_drift']]
        if not non_drift.empty:
            ax.scatter(non_drift['iteration'], non_drift[metric_col], color='blue', s=18, label='Non-drift refresh')
        if not drift.empty:
            ax.scatter(drift['iteration'], drift[metric_col], color='red', s=30, label='Drift-triggered refresh')
        ax.legend(loc='best')
        ax.set_title(title)
        ax.set_xlabel('Iteration')
        ax.set_ylabel(y_label)
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def save_ratio_over_time_plot(trace_df, metric_col, output_path, title, y_label, annotation_text=None):
    # Plot ratio values continuously and highlight only retraining events.
    fig, ax = plt.subplots(figsize=(10, 4.5))
    if trace_df.empty:
        ax.set_title(title)
        ax.set_xlabel('Iteration')
        ax.set_ylabel(y_label)
        ax.text(0.5, 0.5, 'No embedding refresh events', ha='center', va='center', transform=ax.transAxes)
    else:
        ordered = trace_df.sort_values('iteration').reset_index(drop=True)
        ax.plot(ordered['iteration'], ordered[metric_col], color='steelblue', linewidth=1.5)
        retrained = ordered[ordered['retrained_ocsvm']]
        retrained_non_drift = retrained[~retrained['is_drift']]
        retrained_drift = retrained[retrained['is_drift']]
        if not retrained_non_drift.empty:
            ax.scatter(
                retrained_non_drift['iteration'],
                retrained_non_drift[metric_col],
                color='blue',
                s=22,
                label='Retrain after new triple',
            )
        if not retrained_drift.empty:
            ax.scatter(
                retrained_drift['iteration'],
                retrained_drift[metric_col],
                color='red',
                s=28,
                label='Retrain after drift',
            )
        if not retrained.empty:
            ax.legend(loc='best')
        ax.set_title(title)
        ax.set_xlabel('Iteration')
        ax.set_ylabel(y_label)
        if annotation_text:
            ax.text(
                1.02,
                0.98,
                annotation_text,
                transform=ax.transAxes,
                ha='left',
                va='top',
                fontsize=9,
                bbox={'boxstyle': 'round', 'facecolor': 'white', 'alpha': 0.85},
            )
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def build_ratio_trace_records(shift_df, metric_col):
    # Export ratio traces with retraining and drift labels.
    if shift_df.empty:
        return []
    cols = ['iteration', metric_col, 'retrained_ocsvm', 'is_drift', 'trigger_reason']
    trace = shift_df.loc[:, cols].copy().rename(columns={metric_col: 'ratio_value'})
    trace['iteration'] = trace['iteration'].astype(int)
    trace['retrained_ocsvm'] = trace['retrained_ocsvm'].astype(bool)
    trace['is_drift'] = trace['is_drift'].astype(bool)
    return trace.to_dict('records')


def boundary_from_min_max(min_point, max_point):
    # Build one boundary object from min/max feature corners.
    if min_point is None or max_point is None:
        return None
    min_point = np.asarray(min_point, dtype=np.float64).copy()
    max_point = np.asarray(max_point, dtype=np.float64).copy()
    center = 0.5 * (min_point + max_point)
    diagonal_vector = max_point - min_point
    diagonal_length = float(np.linalg.norm(diagonal_vector))
    radius = float(diagonal_length / 2.0)
    return {
        'min_point': min_point,
        'max_point': max_point,
        'center': center,
        'radius': radius,
        'diagonal_length': diagonal_length,
    }


def compute_feature_space_boundary(reference_features_df, feature_cols):
    # Build the triple-level min/max box, center, and radius from the current feature space.
    if reference_features_df.empty:
        return None
    X = reference_features_df.loc[:, feature_cols].to_numpy(dtype=np.float64)
    min_point = X.min(axis=0)
    max_point = X.max(axis=0)
    return boundary_from_min_max(min_point, max_point)


def compute_feature_space_boundary_from_kg(kg_df, embedding_state, compose_fn):
    # Build the same min/max box directly from KG triples without creating a full feature DataFrame.
    min_point = None
    max_point = None
    for kg_row in kg_df[['subject', 'edge_type', 'object']].itertuples(index=False):
        row = {'subject': kg_row.subject, 'edge_type': kg_row.edge_type, 'object': kg_row.object}
        vec = np.asarray(compose_fn(row, embedding_state), dtype=np.float64)
        if min_point is None:
            min_point = vec.copy()
            max_point = vec.copy()
        else:
            min_point = np.minimum(min_point, vec)
            max_point = np.maximum(max_point, vec)
    return boundary_from_min_max(min_point, max_point)


def update_boundary_incrementally(boundary, point_record, feature_cols):
    # Update a saved boundary using one new triple feature while embeddings stay fixed.
    if boundary is None:
        point_vec = np.asarray([point_record[col] for col in feature_cols], dtype=np.float64)
        return boundary_from_min_max(point_vec, point_vec), True
    point_vec = np.asarray([point_record[col] for col in feature_cols], dtype=np.float64)
    min_point = boundary['min_point']
    max_point = boundary['max_point']
    new_min_point = np.minimum(min_point, point_vec)
    new_max_point = np.maximum(max_point, point_vec)
    changed = bool(not np.array_equal(new_min_point, min_point) or not np.array_equal(new_max_point, max_point))
    if not changed:
        return boundary, False
    return boundary_from_min_max(new_min_point, new_max_point), True


def compute_center_radius_metrics(point_record, boundary, feature_cols):
    # Measure how far a triple point is from the current feature-space center.
    if boundary is None:
        return {
            'distance_to_center': np.nan,
            'center_radius': np.nan,
            'center_diagonal_length': np.nan,
            'center_radius_ratio': np.nan,
            'threshold_exceeded_center_radius': False,
        }
    point_vec = np.asarray([point_record[col] for col in feature_cols], dtype=np.float64)
    distance_to_center = float(np.linalg.norm(point_vec - boundary['center']))
    radius = float(boundary['radius'])
    diagonal_length = float(boundary['diagonal_length'])
    if radius == 0.0:
        center_radius_ratio = np.inf if distance_to_center > 0 else 0.0
    else:
        center_radius_ratio = float(distance_to_center / radius)
    return {
        'distance_to_center': distance_to_center,
        'center_radius': radius,
        'center_diagonal_length': diagonal_length,
        'center_radius_ratio': center_radius_ratio,
        'threshold_exceeded_center_radius': bool(np.isfinite(center_radius_ratio) and center_radius_ratio > 1.0),
    }


def add_center_radius_metrics(shift_record, reference_features_df, point_record, feature_cols):
    # Attach triple-level center/radius metrics to an embedding refresh event.
    boundary = compute_feature_space_boundary(reference_features_df, feature_cols)
    shift_record.update(compute_center_radius_metrics(point_record, boundary, feature_cols))
    return shift_record


def compute_retrain_ratio_summary(shift_df, processed_triples):
    # Summarize how often retraining happened for unseen refreshes and overall stream steps.
    if shift_df.empty:
        return {
            'num_unseen_refresh_events': 0,
            'num_unseen_retrains': 0,
            'unseen_retrain_ratio': 0.0,
            'num_all_retrains': 0,
            'processed_future_triples': int(processed_triples),
            'overall_retrain_ratio': 0.0,
        }
    unseen_mask = shift_df['trigger_reason'].eq('unseen_element')
    num_unseen_refresh_events = int(unseen_mask.sum())
    num_unseen_retrains = int(shift_df.loc[unseen_mask, 'retrained_ocsvm'].fillna(False).astype(bool).sum())
    num_all_retrains = int(shift_df['retrained_ocsvm'].fillna(False).astype(bool).sum())
    unseen_retrain_ratio = float(num_unseen_retrains / num_unseen_refresh_events) if num_unseen_refresh_events else 0.0
    overall_retrain_ratio = float(num_all_retrains / processed_triples) if processed_triples else 0.0
    return {
        'num_unseen_refresh_events': num_unseen_refresh_events,
        'num_unseen_retrains': num_unseen_retrains,
        'unseen_retrain_ratio': unseen_retrain_ratio,
        'num_all_retrains': num_all_retrains,
        'processed_future_triples': int(processed_triples),
        'overall_retrain_ratio': overall_retrain_ratio,
    }


def format_retrain_ratio_annotation(summary):
    # Format retraining ratios for display beside the ratio plots.
    return (
        f"unseen retrain ratio: {summary['num_unseen_retrains']} / {summary['num_unseen_refresh_events']} = {summary['unseen_retrain_ratio']:.3f}\n"
        f"overall retrain ratio: {summary['num_all_retrains']} / {summary['processed_future_triples']} = {summary['overall_retrain_ratio']:.3f}"
    )


def save_avg_distance_plot(trace_df, output_path, title):
    # Plot average embedding movement over algorithm iterations.
    save_metric_over_time_plot(trace_df, 'avg_euclidean_distance', output_path, title, 'Average Euclidean Distance')


def save_distance_radius_plot(trace_df, output_path, title):
    # Plot triple-to-center distances as vertical candles with the radius line.
    fig, ax = plt.subplots(figsize=(10, 4.5))
    if trace_df.empty:
        ax.set_title(title)
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Distance / Radius')
        ax.text(0.5, 0.5, 'No center-radius events', ha='center', va='center', transform=ax.transAxes)
    else:
        ordered = trace_df.sort_values('iteration').reset_index(drop=True)
        ax.vlines(ordered['iteration'], ymin=0.0, ymax=ordered['distance_to_center'], color='steelblue', linewidth=1.4)
        ax.plot(ordered['iteration'], ordered['center_radius'], color='darkorange', linewidth=1.8, label='Radius')
        ax.set_title(title)
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Distance / Radius')
        ax.legend(loc='best')
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def save_center_ratio_step_plot(trace_df, output_path, title, annotation_text=None):
    # Plot the center-radius ratio as a step line and highlight retraining events.
    fig, ax = plt.subplots(figsize=(10, 4.5))
    if trace_df.empty:
        ax.set_title(title)
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Center-Radius Ratio')
        ax.text(0.5, 0.5, 'No center-radius events', ha='center', va='center', transform=ax.transAxes)
    else:
        ordered = trace_df.sort_values('iteration').reset_index(drop=True)
        ax.step(ordered['iteration'], ordered['center_radius_ratio'], where='post', color='steelblue', linewidth=1.5)
        retrained = ordered[ordered['retrained_ocsvm']]
        retrained_non_drift = retrained[~retrained['is_drift']]
        retrained_drift = retrained[retrained['is_drift']]
        if not retrained_non_drift.empty:
            ax.scatter(retrained_non_drift['iteration'], retrained_non_drift['center_radius_ratio'], color='blue', s=22, label='Retrain after new triple')
        if not retrained_drift.empty:
            ax.scatter(retrained_drift['iteration'], retrained_drift['center_radius_ratio'], color='red', s=28, label='Retrain after drift')
        if not retrained.empty:
            ax.legend(loc='best')
        ax.set_title(title)
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Center-Radius Ratio')
        if annotation_text:
            ax.text(
                1.02,
                0.98,
                annotation_text,
                transform=ax.transAxes,
                ha='left',
                va='top',
                fontsize=9,
                bbox={'boxstyle': 'round', 'facecolor': 'white', 'alpha': 0.85},
            )
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def build_center_radius_trace_records(shift_df):
    # Export center-radius traces with retraining and drift labels.
    if shift_df.empty:
        return []
    cols = [
        'iteration',
        'distance_to_center',
        'center_radius',
        'center_radius_ratio',
        'retrained_ocsvm',
        'is_drift',
        'trigger_reason',
    ]
    trace = shift_df.loc[:, cols].copy()
    trace['iteration'] = trace['iteration'].astype(int)
    trace['retrained_ocsvm'] = trace['retrained_ocsvm'].astype(bool)
    trace['is_drift'] = trace['is_drift'].astype(bool)
    return trace.to_dict('records')


In [5]:

# def process_triple_for_features(row, kg_df, lookup, embedding_state, compose_fn, feature_cols, composition_name, iteration, retrain_threshold):
#     # Add the new triple, then refresh TransE embeddings only if a new element appeared.
#     has_unseen = triple_has_unseen_elements(row, lookup)
#     kg_df, lookup = update_knowledge_graph(kg_df, pd.DataFrame([row]), lookup)
#     shift_record = None
#     if has_unseen:
#         old_state = embedding_state
#         embedding_state = generate_transe_embeddings_from_kg(kg_df, embedding_size=EMBEDDING_DIM, random_state=EMBEDDING_RANDOM_STATE)
#         shift_record = build_embedding_shift_record(
#             old_state,
#             embedding_state,
#             row,
#             composition_name,
#             iteration,
#             trigger_reason='unseen_element',
#             is_drift=False,
#             threshold=retrain_threshold,
#         )
#     record = row_to_feature_record(row, embedding_state, compose_fn, feature_cols)
#     if shift_record is not None:
#         kg_features_df = rows_to_feature_frame(kg_df, embedding_state, compose_fn, feature_cols)
#         shift_record = add_center_radius_metrics(shift_record, kg_features_df, record, feature_cols)
#     return record, kg_df, lookup, embedding_state, shift_record, has_unseen

def process_triple_for_features(row, kg_df, lookup, embedding_state, current_boundary, compose_fn, feature_cols, composition_name, iteration, retrain_threshold):
    # Keep the boundary live while embeddings stay fixed, and rebuild it only after TransE refreshes.
    has_unseen = triple_has_unseen_elements(row, lookup)
    shift_record = None
    if not has_unseen:
        record = row_to_feature_record(row, embedding_state, compose_fn, feature_cols)
        kg_df, lookup = update_knowledge_graph(kg_df, pd.DataFrame([row]), lookup)
        current_boundary, _ = update_boundary_incrementally(current_boundary, record, feature_cols)
        return record, kg_df, lookup, embedding_state, current_boundary, shift_record, has_unseen

    pre_unseen_boundary = current_boundary
    kg_df, lookup = update_knowledge_graph(kg_df, pd.DataFrame([row]), lookup)
    write_progress('reembedding_unseen_start', iteration, composition=composition_name, detail='training TransE on updated KG')
    embedding_state = generate_transe_embeddings_from_kg(
        kg_df,
        embedding_size=EMBEDDING_DIM,
        random_state=EMBEDDING_RANDOM_STATE,
    )
    write_progress('reembedding_unseen_done', iteration, composition=composition_name, detail='TransE refresh complete')
    shift_record = build_center_radius_shift_record(
        row=row,
        composition_name=composition_name,
        iteration=iteration,
        trigger_reason='unseen_element',
        is_drift=False,
        threshold=retrain_threshold,
        kg_rows=len(kg_df),
    )
    record = row_to_feature_record(row, embedding_state, compose_fn, feature_cols)
    write_progress('boundary_compare_unseen_start', iteration, composition=composition_name, detail='comparing post-refresh triple to saved boundary')
    shift_record.update(compute_center_radius_metrics(record, pre_unseen_boundary, feature_cols))
    write_progress('boundary_compare_unseen_done', iteration, composition=composition_name, detail='saved-boundary ratio ready')
    write_progress('boundary_rebuild_unseen_start', iteration, composition=composition_name, detail='rebuilding KG boundary after TransE refresh')
    current_boundary = compute_feature_space_boundary_from_kg(kg_df, embedding_state, compose_fn)
    write_progress('boundary_rebuild_unseen_done', iteration, composition=composition_name, detail='updated KG boundary ready')
    return record, kg_df, lookup, embedding_state, current_boundary, shift_record, has_unseen


# def rebuild_reference_after_drift(reference_raw_df, kg_df, lookup, embedding_state, compose_fn, feature_cols, composition_name, iteration, drift_row, retrain_threshold):
#     # After drift, regenerate TransE embeddings from current KG and rebuild reference features.
#     old_state = embedding_state
#     embedding_state = generate_transe_embeddings_from_kg(kg_df, embedding_size=EMBEDDING_DIM, random_state=EMBEDDING_RANDOM_STATE)
#     shift_record = build_embedding_shift_record(
#         old_state,
#         embedding_state,
#         drift_row,
#         composition_name,
#         iteration,
#         trigger_reason='drift',
#         is_drift=True,
#         threshold=retrain_threshold,
#     )
#     reference_features_df = rows_to_feature_frame(reference_raw_df, embedding_state, compose_fn, feature_cols)
#     drift_feature_record = row_to_feature_record(drift_row, embedding_state, compose_fn, feature_cols)
#     kg_features_df = rows_to_feature_frame(kg_df, embedding_state, compose_fn, feature_cols)
#     shift_record = add_center_radius_metrics(shift_record, kg_features_df, drift_feature_record, feature_cols)
#     return reference_features_df, kg_df, lookup, embedding_state, shift_record
    
def rebuild_reference_after_drift(reference_raw_df, kg_df, lookup, embedding_state, compose_fn, feature_cols, composition_name, iteration, drift_row, retrain_threshold):
    # After drift, regenerate TransE embeddings from current KG and rebuild reference features.
    write_progress('reembedding_drift_start', iteration, composition=composition_name, detail='training TransE after drift')
    embedding_state = generate_transe_embeddings_from_kg(
        kg_df,
        embedding_size=EMBEDDING_DIM,
        random_state=EMBEDDING_RANDOM_STATE,
    )
    write_progress('reembedding_drift_done', iteration, composition=composition_name, detail='post-drift TransE ready')
    shift_record = build_center_radius_shift_record(
        row=drift_row,
        composition_name=composition_name,
        iteration=iteration,
        trigger_reason='drift',
        is_drift=True,
        threshold=retrain_threshold,
        kg_rows=len(kg_df),
    )
    write_progress('reference_rebuild_start', iteration, composition=composition_name, detail='rebuilding reference features')
    reference_features_df = rows_to_feature_frame(reference_raw_df, embedding_state, compose_fn, feature_cols)
    write_progress('reference_rebuild_done', iteration, composition=composition_name, detail='reference features rebuilt')
    drift_feature_record = row_to_feature_record(drift_row, embedding_state, compose_fn, feature_cols)
    write_progress('boundary_scan_drift_start', iteration, composition=composition_name, detail='scanning KG min/max bounds after drift')
    boundary = compute_feature_space_boundary_from_kg(kg_df, embedding_state, compose_fn)
    shift_record.update(compute_center_radius_metrics(drift_feature_record, boundary, feature_cols))
    write_progress('boundary_scan_drift_done', iteration, composition=composition_name, detail='post-drift center-radius metrics ready')
    return reference_features_df, kg_df, lookup, embedding_state, boundary, shift_record

def detect_drift(detector, current_flags):
    # Preserve the existing Z-test drift detector behavior.
    return detector.detect(current_flags)


def run_dynamic_ocsvm_ztest_stream_keep10(
    initial_kg,
    initial_stream,
    future_stream,
    lookup,
    compose_fn,
    feature_cols,
    composition_name,
    window_size=WINDOW_SIZE,
    nu=NU,
    gamma=GAMMA,
    alpha=ALPHA,
    standardize=STANDARDIZE,
    keep_fraction_after_drift=KEEP_FRACTION_AFTER_DRIFT,
    embedding_shift_retrain_threshold=EMBEDDING_SHIFT_RETRAIN_THRESHOLD,
):
    # Initial embeddings are generated only from KG known through the threshold year.
    if initial_stream.empty:
        raise ValueError(f'No initial training triples for {composition_name}.')
    if future_stream.empty:
        raise ValueError(f'No future stream triples for {composition_name}.')

    kg_df = initial_kg.copy().reset_index(drop=True)
    lookup = {'nodes': set(lookup['nodes']), 'edge_types': set(lookup['edge_types'])}
    write_progress('initial_embedding_start', 0, len(future_stream), composition=composition_name, detail='training initial TransE from cutoff KG')
    embedding_state = generate_transe_embeddings_from_kg(kg_df, embedding_size=EMBEDDING_DIM, random_state=EMBEDDING_RANDOM_STATE)
    write_progress('initial_embedding_done', 0, len(future_stream), composition=composition_name, detail='initial TransE ready')
    write_progress('boundary_init_start', 0, len(future_stream), composition=composition_name, detail='building initial KG boundary')
    current_boundary = compute_feature_space_boundary_from_kg(kg_df, embedding_state, compose_fn)
    write_progress('boundary_init_done', 0, len(future_stream), composition=composition_name, detail='initial KG boundary ready')

    if len(initial_stream) < 2:
        raise ValueError(f'Need at least 2 initial triples for {composition_name} to separate training and monitoring windows.')

    seed_window_size = min(window_size, len(initial_stream) - 1)
    reference_raw_df = initial_stream.iloc[:-seed_window_size].copy().reset_index(drop=True)
    initial_window_raw_df = initial_stream.iloc[-seed_window_size:].copy().reset_index(drop=True)
    initial_features_df = rows_to_feature_frame(reference_raw_df, embedding_state, compose_fn, feature_cols)
    labeled_initial_reference, ocsvm, scaler, reference_flags = train_classifier(
        initial_features_df,
        feature_cols=feature_cols,
        nu=nu,
        gamma=gamma,
        standardize=standardize,
    )

    detector = ZTestDriftDetector(alpha=alpha)
    detector.reset(reference_flags)

    current_raw_window = deque(
        initial_window_raw_df.to_dict('records'),
        maxlen=window_size,
    )
    future_records = future_stream.to_dict('records')

    embedding_shift_records = []
    threshold_retrain_count = 0
    unseen_elements_count_all = 0
    i = 0
    iteration = 0
    step_records = []
    progress = tqdm(total=len(future_records), desc=f'{composition_name} stream', unit='triple')
    write_progress('starting', 0, len(future_records), 0, 0, 0, composition_name)

    while len(current_raw_window) < window_size and i < len(future_records):
        row = pd.Series(future_records[i])
        i += 1
        progress.update(1)
        iteration += 1
        warmup_record, kg_df, lookup, embedding_state, current_boundary, shift_record, has_unseen = process_triple_for_features(
            row, kg_df, lookup, embedding_state, current_boundary, compose_fn, feature_cols, composition_name, iteration, embedding_shift_retrain_threshold
        )
        if has_unseen:
            unseen_elements_count_all += 1
        should_retrain_from_embedding_shift = False
        if shift_record is not None:
            # should_retrain_from_embedding_shift = shift_record['threshold_exceeded_avg']
            # should_retrain_from_embedding_shift = shift_record['threshold_exceeded_max_spread']
            # should_retrain_from_embedding_shift = shift_record['threshold_exceeded_min_spread']
            should_retrain_from_embedding_shift = shift_record['threshold_exceeded_center_radius']
        if shift_record is not None and should_retrain_from_embedding_shift:
            _, ocsvm, scaler, reference_flags = retrain_reference_model(
                reference_raw_df,
                embedding_state,
                compose_fn,
                feature_cols,
                detector,
                nu=nu,
                gamma=gamma,
                standardize=standardize,
                iteration=iteration,
                total=len(future_records),
                composition_name=composition_name,
            )
            shift_record['retrained_ocsvm'] = True
            threshold_retrain_count += 1
        if shift_record is not None:
            embedding_shift_records.append(shift_record)
        current_raw_window.append(row.to_dict())
        step_records.append({
            'iteration': int(iteration),
            'stream_position': int(warmup_record['stream_position']),
            'row_number': int(warmup_record['row_number']),
            'timestamp': warmup_record['timestamp'],
            'type': warmup_record['type'],
            'subject': warmup_record['subject'],
            'edge_type': warmup_record['edge_type'],
            'object': warmup_record['object'],
            'composition': composition_name,
            'drift_tested': False,
            'drift_detected': False,
            'reference_outlier_fraction': np.nan,
            'current_outlier_fraction': np.nan,
            'embedding_kg_rows': int(embedding_state.kg_rows),
        })
        progress.set_postfix({'phase': 'warmup', 'drifts': 0, 'refreshes': len(embedding_shift_records), 'unseen_all': unseen_elements_count_all})
        write_progress('warmup', i, len(future_records), 0, len(embedding_shift_records), unseen_elements_count_all, composition_name)

    first_drift_detected = False
    drift_records = []
    drift_windows = []
    window_history = []

    while i < len(future_records):
        row = pd.Series(future_records[i])
        i += 1
        progress.update(1)
        iteration += 1
        feature_record, kg_df, lookup, embedding_state, current_boundary, shift_record, has_unseen = process_triple_for_features(
            row, kg_df, lookup, embedding_state, current_boundary, compose_fn, feature_cols, composition_name, iteration, embedding_shift_retrain_threshold
        )
        if has_unseen:
            unseen_elements_count_all += 1
        should_retrain_from_embedding_shift = False
        if shift_record is not None:
            # should_retrain_from_embedding_shift = shift_record['threshold_exceeded_avg']
            # should_retrain_from_embedding_shift = shift_record['threshold_exceeded_max_spread']
            # should_retrain_from_embedding_shift = shift_record['threshold_exceeded_min_spread']
            should_retrain_from_embedding_shift = shift_record['threshold_exceeded_center_radius']
        if shift_record is not None and should_retrain_from_embedding_shift:
            _, ocsvm, scaler, reference_flags = retrain_reference_model(
                reference_raw_df,
                embedding_state,
                compose_fn,
                feature_cols,
                detector,
                nu=nu,
                gamma=gamma,
                standardize=standardize,
                iteration=iteration,
                total=len(future_records),
                composition_name=composition_name,
            )
            shift_record['retrained_ocsvm'] = True
            threshold_retrain_count += 1
        if shift_record is not None:
            embedding_shift_records.append(shift_record)
        current_raw_window.append(row.to_dict())

        current_raw_df = pd.DataFrame(list(current_raw_window)).reset_index(drop=True)
        current_features_df = rows_to_feature_frame(current_raw_df, embedding_state, compose_fn, feature_cols)
        X_curr, _ = build_feature_matrix(current_features_df, feature_cols, scaler=scaler, fit=False)
        current_flags = outlier_flags_from_ocsvm(ocsvm, X_curr)
        is_drift = detect_drift(detector, current_flags)

        stream_position = int(feature_record['stream_position'])
        row_number = int(feature_record['row_number'])
        timestamp_value = feature_record['timestamp']
        type_value = feature_record['type']
        current_outlier_fraction = float(np.mean(current_flags)) if len(current_flags) else 0.0
        reference_outlier_fraction = float(reference_flags.mean()) if len(reference_flags) else 0.0

        window_history.append({
            'iteration': int(iteration),
            'stream_position': stream_position,
            'row_number': row_number,
            'timestamp': timestamp_value,
            'type': type_value,
            'composition': composition_name,
            'window_size': int(len(current_features_df)),
            'reference_size': int(len(reference_flags)),
            'reference_outlier_fraction': reference_outlier_fraction,
            'current_outlier_fraction': current_outlier_fraction,
            'drift_detected': bool(is_drift),
            'embedding_kg_rows': int(embedding_state.kg_rows),
        })
        step_records.append({
            'iteration': int(iteration),
            'stream_position': stream_position,
            'row_number': row_number,
            'timestamp': timestamp_value,
            'type': type_value,
            'subject': feature_record['subject'],
            'edge_type': feature_record['edge_type'],
            'object': feature_record['object'],
            'composition': composition_name,
            'drift_tested': True,
            'drift_detected': bool(is_drift),
            'reference_outlier_fraction': reference_outlier_fraction,
            'current_outlier_fraction': current_outlier_fraction,
            'embedding_kg_rows': int(embedding_state.kg_rows),
        })
        progress.set_postfix({'phase': 'monitor', 'drifts': len(drift_records), 'refreshes': len(embedding_shift_records), 'unseen_all': unseen_elements_count_all})
        write_progress('monitor', i, len(future_records), len(drift_records), len(embedding_shift_records), unseen_elements_count_all, composition_name)

        if not is_drift:
            continue

        drift_event_id = len(drift_records) + 1
        drift_windows.append(build_labeled_window_snapshot(current_features_df, current_flags, drift_event_id, stream_position))
        drift_records.append({
            'iteration': int(iteration),
            'drift_event_id': drift_event_id,
            'stream_position': stream_position,
            'row_number': row_number,
            'timestamp': timestamp_value,
            'type': type_value,
            'composition': composition_name,
            'window_size': int(len(current_features_df)),
            'reference_size': int(len(reference_flags)),
            'reference_outlier_fraction': reference_outlier_fraction,
            'current_outlier_fraction': current_outlier_fraction,
            'first_drift_phase': not first_drift_detected,
            'embedding_kg_rows': int(embedding_state.kg_rows),
        })
        first_drift_detected = True

        keep_count = max(1, int(round(window_size * keep_fraction_after_drift)))
        retained_raw_rows = list(current_raw_df.tail(min(keep_count, len(current_raw_df))).to_dict('records'))
        rebuilt_raw_rows = list(retained_raw_rows)

        while len(rebuilt_raw_rows) < window_size and i < len(future_records):
            next_row = pd.Series(future_records[i])
            i += 1
            progress.update(1)
            refill_has_unseen = triple_has_unseen_elements(next_row, lookup)
            if refill_has_unseen:
                unseen_elements_count_all += 1
            kg_df, lookup = update_knowledge_graph(kg_df, pd.DataFrame([next_row]), lookup)
            rebuilt_raw_rows.append(next_row.to_dict())
            progress.set_postfix({'phase': 'rebuild', 'drifts': len(drift_records), 'refreshes': len(embedding_shift_records), 'unseen_all': unseen_elements_count_all})
            write_progress('rebuild', i, len(future_records), len(drift_records), len(embedding_shift_records), unseen_elements_count_all, composition_name)

        reference_raw_df = pd.DataFrame(rebuilt_raw_rows).reset_index(drop=True)
        reference_features_df, kg_df, lookup, embedding_state, current_boundary, drift_shift_record = rebuild_reference_after_drift(
            reference_raw_df,
            kg_df,
            lookup,
            embedding_state,
            compose_fn,
            feature_cols,
            composition_name,
            iteration,
            row,
            embedding_shift_retrain_threshold,
        )
        embedding_shift_records.append(drift_shift_record)
        current_raw_window = deque(reference_raw_df.tail(min(window_size, len(reference_raw_df))).to_dict('records'), maxlen=window_size)
        reference_features_df, ocsvm, scaler, reference_flags = retrain_reference_model(
            reference_raw_df,
            embedding_state,
            compose_fn,
            feature_cols,
            detector,
            nu=nu,
            gamma=gamma,
            standardize=standardize,
            iteration=iteration,
            total=len(future_records),
            composition_name=composition_name,
        )
        progress.set_postfix({'phase': 'monitor', 'drifts': len(drift_records), 'refreshes': len(embedding_shift_records), 'unseen_all': unseen_elements_count_all})
        write_progress('monitor', i, len(future_records), len(drift_records), len(embedding_shift_records), unseen_elements_count_all, composition_name)

    write_progress('finished', len(future_records), len(future_records), len(drift_records), len(embedding_shift_records), unseen_elements_count_all, composition_name)
    progress.close()
    embedding_shift_df = pd.DataFrame(embedding_shift_records)
    avg_distance_trace_df = build_avg_distance_trace(embedding_shift_df)
    retrain_ratio_summary = compute_retrain_ratio_summary(embedding_shift_df, processed_triples=i)
    summary = {
        'composition': composition_name,
        'initial_kg_rows': int(len(initial_kg)),
        'initial_reference_size': int(len(initial_features_df)),
        'initial_window_seed_size': int(len(initial_window_raw_df)),
        'future_rows': int(len(future_stream)),
        'num_detected_drifts': int(len(drift_records)),
        'first_drift_stream_position': int(drift_records[0]['stream_position']) if drift_records else None,
        'first_drift_row_number': int(drift_records[0]['row_number']) if drift_records else None,
        'window_size': int(window_size),
        'nu': float(nu),
        'gamma': float(gamma),
        'alpha': float(alpha),
        'standardize': bool(standardize),
        'keep_fraction_after_drift': float(keep_fraction_after_drift),
        'initial_cutoff_year': int(INITIAL_CUTOFF_YEAR),
        'embedding_dim': int(EMBEDDING_DIM),
        'embedding_random_state': int(EMBEDDING_RANDOM_STATE),
        'embedding_policy': 'dynamic_transe_pykeen_from_current_kg_only',
        'embedding_shift_retrain_threshold': float(embedding_shift_retrain_threshold),
        'num_embedding_refresh_events': int(len(embedding_shift_df)),
        'unseen_elements_count_all': int(unseen_elements_count_all),
        'num_center_radius_outside_boundary': int(embedding_shift_df['threshold_exceeded_center_radius'].fillna(False).astype(bool).sum()) if not embedding_shift_df.empty and 'threshold_exceeded_center_radius' in embedding_shift_df.columns else 0,
        'num_threshold_retrains': int(threshold_retrain_count),
        **retrain_ratio_summary,
    }

    drift_columns = [
        'iteration', 'drift_event_id', 'stream_position', 'row_number', 'timestamp', 'type', 'composition',
        'window_size', 'reference_size', 'reference_outlier_fraction', 'current_outlier_fraction',
        'first_drift_phase', 'embedding_kg_rows'
    ]
    history_columns = [
        'iteration', 'stream_position', 'row_number', 'timestamp', 'type', 'composition', 'window_size',
        'reference_size', 'reference_outlier_fraction', 'current_outlier_fraction',
        'drift_detected', 'embedding_kg_rows'
    ]
    return {
        'labeled_reference': labeled_initial_reference,
        'drifts': pd.DataFrame(drift_records, columns=drift_columns),
        'drift_windows': pd.concat(drift_windows, ignore_index=True) if drift_windows else pd.DataFrame(),
        'window_history': pd.DataFrame(window_history, columns=history_columns),
        'step_trace': pd.DataFrame(step_records),
        'embedding_shift_metrics': embedding_shift_df,
        'avg_distance_trace': avg_distance_trace_df,
        'final_kg': kg_df.copy().reset_index(drop=True),
        'summary': summary,
    }


In [ ]:
# Load temporal triples only; precomputed future-leaking embeddings are not loaded.
triples_df = load_temporal_triples(TEMPORAL_TRIPLES_PATH)

# Build initial KG and stream partitions.
initial_kg, initial_stream, future_stream, initial_lookup = build_initial_knowledge_graph(
    triples_df,
    cutoff_year=INITIAL_CUTOFF_YEAR,
    target_type=TARGET_TYPE,
)

print('all temporal triples:', triples_df.shape)
print('initial KG rows:', len(initial_kg))
print('initial stream rows:', len(initial_stream))
print('future stream rows:', len(future_stream))
initial_stream.head()


**Hadamard Flow**


In [ ]:
# Run the Hadamard composition workflow independently.
hadamard_feature_cols = feature_columns(EMBEDDING_DIM)

hadamard_result = run_dynamic_ocsvm_ztest_stream_keep10(
    initial_kg=initial_kg,
    initial_stream=initial_stream,
    future_stream=future_stream,
    lookup=initial_lookup,
    compose_fn=create_hadamard_features,
    feature_cols=hadamard_feature_cols,
    composition_name='hadamard',
    window_size=WINDOW_SIZE,
    nu=NU,
    gamma=GAMMA,
    alpha=ALPHA,  
    standardize=STANDARDIZE,
)
    
hadamard_result['summary']


**Concatenation Flow**


In [ ]:
# # Run the Concatenation composition workflow independently.
# concat_feature_cols = feature_columns(EMBEDDING_DIM * 3)

# concat_result = run_dynamic_ocsvm_ztest_stream_keep10(
#     initial_kg=initial_kg,
#     initial_stream=initial_stream,
#     future_stream=future_stream,
#     lookup=initial_lookup,
#     compose_fn=create_concatenation_features,
#     feature_cols=concat_feature_cols,
#     composition_name='concatenation',
#     window_size=WINDOW_SIZE,
#     nu=NU,
#     gamma=GAMMA,
#     alpha=ALPHA,
#     standardize=STANDARDIZE,
# )

# concat_result['summary']


In [ ]:

def safe_run_name(name):
    return str(name).replace(' - ', '_-_').replace(' ', '_').replace('/', '_').replace('\\', '_')


RUN_NAME = safe_run_name(TARGET_TYPE if TARGET_TYPE is not None else 'all_types')
run_dir = RESULTS_DIR / f'{RUN_NAME}_temporal_dynamic_embeddings_keep10_after_drift'
run_dir.mkdir(parents=True, exist_ok=True)

hadamard_result['labeled_reference'].to_csv(run_dir / 'hadamard_initial_reference_labels.tsv', sep='	', index=False)
hadamard_result['drifts'].to_csv(run_dir / 'hadamard_detected_drifts.tsv', sep='	', index=False)
hadamard_result['window_history'].to_csv(run_dir / 'hadamard_window_history.tsv', sep='	', index=False)
hadamard_result['drift_windows'].to_csv(run_dir / 'hadamard_drift_windows.tsv', sep='	', index=False)
hadamard_result['step_trace'].to_csv(run_dir / 'hadamard_step_trace.tsv', sep='	', index=False)
hadamard_result['embedding_shift_metrics'].to_csv(run_dir / 'hadamard_embedding_shift_metrics.tsv', sep='	', index=False)
hadamard_result['avg_distance_trace'].to_csv(run_dir / 'hadamard_avg_distance_trace.tsv', sep='	', index=False)
hadamard_result['final_kg'].to_csv(run_dir / 'hadamard_final_kg.tsv', sep='	', index=False)
save_avg_distance_plot(
    hadamard_result['avg_distance_trace'],
    run_dir / 'hadamard_avg_distance_plot.png',
    'Hadamard Embedding Shift Across Iterations',
)
save_metric_over_time_plot(
    hadamard_result['embedding_shift_metrics'],
    'max_coordinate_spread',
    run_dir / 'hadamard_max_coordinate_spread_plot.png',
    'Hadamard Max Coordinate Spread Across Iterations',
    'Max Coordinate Spread',
)
save_metric_over_time_plot(
    hadamard_result['embedding_shift_metrics'],
    'min_coordinate_spread',
    run_dir / 'hadamard_min_coordinate_spread_plot.png',
    'Hadamard Min Coordinate Spread Across Iterations',
    'Min Coordinate Spread',
)
save_ratio_over_time_plot(
    hadamard_result['embedding_shift_metrics'],
    'decision_distance_max_spread',
    run_dir / 'hadamard_max_spread_ratio_plot.png',
    'Hadamard Max-Spread Ratio Across Iterations',
    'Max-Spread Ratio',
    annotation_text=format_retrain_ratio_annotation(hadamard_result['summary']),
)
save_ratio_over_time_plot(
    hadamard_result['embedding_shift_metrics'],
    'decision_distance_min_spread',
    run_dir / 'hadamard_min_spread_ratio_plot.png',
    'Hadamard Min-Spread Ratio Across Iterations',
    'Min-Spread Ratio',
    annotation_text=format_retrain_ratio_annotation(hadamard_result['summary']),
)
save_distance_radius_plot(
    hadamard_result['embedding_shift_metrics'],
    run_dir / 'hadamard_distance_to_center_radius_plot.png',
    'Hadamard Distance to Center and Radius Across Iterations',
)
save_center_ratio_step_plot(
    hadamard_result['embedding_shift_metrics'],
    run_dir / 'hadamard_center_radius_ratio_plot.png',
    'Hadamard Center-Radius Ratio Across Iterations',
    annotation_text=format_retrain_ratio_annotation(hadamard_result['summary']),
)
with (run_dir / 'hadamard_max_spread_ratio_trace.json').open('w', encoding='utf-8') as f:
    json.dump(build_ratio_trace_records(hadamard_result['embedding_shift_metrics'], 'decision_distance_max_spread'), f, indent=2)
with (run_dir / 'hadamard_min_spread_ratio_trace.json').open('w', encoding='utf-8') as f:
    json.dump(build_ratio_trace_records(hadamard_result['embedding_shift_metrics'], 'decision_distance_min_spread'), f, indent=2)
pd.DataFrame(build_center_radius_trace_records(hadamard_result['embedding_shift_metrics'])).to_csv(
    run_dir / 'hadamard_center_radius_trace.tsv',
    sep='\t',
    index=False,
)
with (run_dir / 'hadamard_center_radius_trace.json').open('w', encoding='utf-8') as f:
    json.dump(build_center_radius_trace_records(hadamard_result['embedding_shift_metrics']), f, indent=2)

# concat_result['labeled_reference'].to_csv(run_dir / 'concatenation_initial_reference_labels.tsv', sep='	', index=False)
# concat_result['drifts'].to_csv(run_dir / 'concatenation_detected_drifts.tsv', sep='	', index=False)
# concat_result['window_history'].to_csv(run_dir / 'concatenation_window_history.tsv', sep='	', index=False)
# concat_result['drift_windows'].to_csv(run_dir / 'concatenation_drift_windows.tsv', sep='	', index=False)
# concat_result['step_trace'].to_csv(run_dir / 'concatenation_step_trace.tsv', sep='	', index=False)
# concat_result['embedding_shift_metrics'].to_csv(run_dir / 'concatenation_embedding_shift_metrics.tsv', sep='	', index=False)
# concat_result['avg_distance_trace'].to_csv(run_dir / 'concatenation_avg_distance_trace.tsv', sep='	', index=False)
# concat_result['final_kg'].to_csv(run_dir / 'concatenation_final_kg.tsv', sep='	', index=False)
# save_avg_distance_plot(
#     concat_result['avg_distance_trace'],
#     run_dir / 'concatenation_avg_distance_plot.png',
#     'Concatenation Embedding Shift Across Iterations',
# )
# save_metric_over_time_plot(
#     concat_result['embedding_shift_metrics'],
#     'max_coordinate_spread',
#     run_dir / 'concatenation_max_coordinate_spread_plot.png',
#     'Concatenation Max Coordinate Spread Across Iterations',
#     'Max Coordinate Spread',
# )
# save_metric_over_time_plot(
#     concat_result['embedding_shift_metrics'],
#     'min_coordinate_spread',
#     run_dir / 'concatenation_min_coordinate_spread_plot.png',
#     'Concatenation Min Coordinate Spread Across Iterations',
#     'Min Coordinate Spread',
# )
# save_ratio_over_time_plot(
#     concat_result['embedding_shift_metrics'],
#     'decision_distance_max_spread',
#     run_dir / 'concatenation_max_spread_ratio_plot.png',
#     'Concatenation Max-Spread Ratio Across Iterations',
#     'Max-Spread Ratio',
# )
# save_ratio_over_time_plot(
#     concat_result['embedding_shift_metrics'],
#     'decision_distance_min_spread',
#     run_dir / 'concatenation_min_spread_ratio_plot.png',
#     'Concatenation Min-Spread Ratio Across Iterations',
#     'Min-Spread Ratio',
#     annotation_text=format_retrain_ratio_annotation(concat_result['summary']),
# )
# save_distance_radius_plot(
#     concat_result['embedding_shift_metrics'],
#     run_dir / 'concatenation_distance_to_center_radius_plot.png',
#     'Concatenation Distance to Center and Radius Across Iterations',
# )
# save_center_ratio_step_plot(
#     concat_result['embedding_shift_metrics'],
#     run_dir / 'concatenation_center_radius_ratio_plot.png',
#     'Concatenation Center-Radius Ratio Across Iterations',
#     annotation_text=format_retrain_ratio_annotation(concat_result['summary']),
# )
# with (run_dir / 'concatenation_max_spread_ratio_trace.json').open('w', encoding='utf-8') as f:
#     json.dump(build_ratio_trace_records(concat_result['embedding_shift_metrics'], 'decision_distance_max_spread'), f, indent=2)
# with (run_dir / 'concatenation_min_spread_ratio_trace.json').open('w', encoding='utf-8') as f:
#     json.dump(build_ratio_trace_records(concat_result['embedding_shift_metrics'], 'decision_distance_min_spread'), f, indent=2)
# pd.DataFrame(build_center_radius_trace_records(concat_result['embedding_shift_metrics'])).to_csv(
#     run_dir / 'concatenation_center_radius_trace.tsv',
#     sep='\t',
#     index=False,
# )
# with (run_dir / 'concatenation_center_radius_trace.json').open('w', encoding='utf-8') as f:
#     json.dump(build_center_radius_trace_records(concat_result['embedding_shift_metrics']), f, indent=2)

with (run_dir / 'summary.json').open('w', encoding='utf-8') as f:
    json.dump(
        {
            'stream_type': TARGET_TYPE,
            'mode': 'temporal_dynamic_embeddings_keep10_after_drift',
            'inputs': {'temporal_triples': str(TEMPORAL_TRIPLES_PATH)},
            'temporal_leakage_note': 'Embeddings are generated from current kg_df only; no precomputed future embeddings are used.',
            'hadamard': hadamard_result['summary'],
            # 'concatenation': concat_result['summary'],
        },
        f,
        indent=2,
    )

print('Saved results to:', run_dir)


In [ ]:
def print_drift_timestamps(label, drifts_df):
    # Print compact drift timestamp summaries.
    if drifts_df.empty:
        print(f'{label}: no drifts detected')
        return
    cols = ['drift_event_id', 'stream_position', 'row_number', 'timestamp', 'current_outlier_fraction']
    print(f'{label}:')
    print(drifts_df[cols].to_string(index=False))


fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, title, result in [
    (axes[0], 'Hadamard - Temporal Dynamic Embeddings Keep 10%', hadamard_result),
    # (axes[1], 'Concatenation - Temporal Dynamic Embeddings Keep 10%', concat_result),
]:
    history = result['window_history']
    drifts = result['drifts']
    if history.empty:
        ax.set_title(f'{title} (no history)')
        ax.set_xlabel('stream position')
        ax.grid(alpha=0.2)
        continue
    ax.plot(history['stream_position'], history['current_outlier_fraction'], label='current outlier fraction')
    ax.plot(history['stream_position'], history['reference_outlier_fraction'], label='reference outlier fraction', alpha=0.8)
    if not drifts.empty:
        ax.scatter(drifts['stream_position'], drifts['current_outlier_fraction'], color='red', s=20, label='detected drifts')
    ax.set_title(title)
    ax.set_xlabel('stream position')
    ax.grid(alpha=0.2)

axes[0].set_ylabel('outlier fraction')
axes[0].legend(loc='best')
plt.tight_layout()

print_drift_timestamps('Hadamard', hadamard_result['drifts'])
# print_drift_timestamps('Concatenation', concat_result['drifts'])
